# SAT-00 · Get the dataset from Roboflow (OPTIONAL)  (CPU, Internet ON)

**You only need this if** you want to (a) prove that your Kaggle copy `rifat4018/satelite-coco` is the same as the
Roboflow version, or (b) build the Kaggle copy again from Roboflow.

**Before running:**
1. Roboflow -> Settings -> API Keys -> copy the *Private API Key*.
2. Kaggle notebook -> Add-ons -> Secrets -> add a secret named `ROBOFLOW_API_KEY`, tick it for this notebook.
3. Open your project page on Roboflow. The URL looks like `app.roboflow.com/<WORKSPACE>/aerial-object-detection-u3f5u/<VERSION>`.
   Put those two values in the settings cell below.

**Attach:** the `sat-ext` code Dataset (+ optionally `satelite-coco` to compare). **Accelerator:** None. **Internet:** ON.
**Produces:** `/kaggle/working/rf_<project>_v<version>/` (train/ valid/ test/ COCO) -> Output tab -> *New Dataset* if you want to use it.

**Input setup** - right panel -> *Add Input* -> search the name -> *Add*. Links (your own Datasets are private: open them from the account that owns them, or share them with this account first):

* `sat-ext` - code (upload the sat_ext folder of the zip): https://www.kaggle.com/datasets/rifat4018/sat-ext
* `satelite-coco` - Roboflow COCO export, 640x640 (or download: universe.roboflow.com/rifat4018/aerial-object-detection-u3f5u): https://www.kaggle.com/datasets/rifat4018/satelite-coco
* Check the printed input tree in the next cell before running the rest.

In [ ]:
import os

def show_inputs(depth=4, max_files_per_dir=6):
    """Print what is attached under /kaggle/input - check it before trusting the rest."""
    root0 = os.environ.get("SAT_INPUT", "/kaggle/input")
    for root, dirs, files in os.walk(root0):
        dirs[:] = sorted(d for d in dirs if not d.startswith("."))
        level = root.replace(root0, "").count(os.sep)
        if level > depth:
            dirs[:] = []
            continue
        print("  " * level + root + "/")
        for fn in sorted(files)[:max_files_per_dir]:
            print("  " * level + "  " + fn)
        if len(files) > max_files_per_dir:
            print("  " * level + f"  ... +{len(files) - max_files_per_dir} more files")

show_inputs()

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
INPUT_ROOT = os.environ.get("SAT_INPUT", "/kaggle/input")

def _find_code_root():
    for root, dirs, files in os.walk(INPUT_ROOT):
        if {"config.py", "audit.py", "metrics.py", "infer.py"} <= set(files):
            return root
    return None

CODE_ROOT = os.environ.get("SAT_CODE") or _find_code_root()
assert CODE_ROOT, "sat_ext code folder not found - Add Input -> your 'sat-ext' code Dataset"
sys.path.insert(0, CODE_ROOT)
print("CODE_ROOT:", CODE_ROOT)

import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 70)
import config as cfg, utils
utils.ensure_dirs(); utils.set_seed(cfg.SEED)
print("work dir:", cfg.WORK_DIR, "| smoke test:", cfg.SMOKE)

In [ ]:
RF_WORKSPACE = cfg.RF_WORKSPACE      # CHECK: the part after app.roboflow.com/ in your project URL
RF_PROJECT   = cfg.RF_PROJECT        # "aerial-object-detection-u3f5u"
RF_VERSION   = cfg.RF_VERSION        # CHECK: the version number you generated (Versions tab)
print(RF_WORKSPACE, RF_PROJECT, RF_VERSION)

In [ ]:
utils.warn_if_gpu_wasted()
import roboflow_io as rio, dataset as ds
dest = rio.download(RF_WORKSPACE, RF_PROJECT, RF_VERSION)

In [ ]:
# compare with the Kaggle copy if it is attached (the copy your paper used)
others = [r for r in ds.find_coco_roots() if not r.startswith(cfg.WORK_DIR)]
if others:
    table, identical = rio.compare_exports(others[0], dest)
    table.to_csv(os.path.join(cfg.WORK_DIR, "roboflow_vs_kaggle.csv"), index=False)
else:
    print("no Kaggle copy attached -> nothing to compare. The download is in", dest)